In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# df = pd.read_csv('dataset.csv') 


data = {
    'text': [
        "I loved this movie, it was fantastic!", "The service was terrible and slow.",
        "It was an okay experience, nothing special.", "Absolutely brilliant acting!",
        "I will never buy this again, waste of money.", "The food was mediocre at best."
    ],
    'sentiment': ['Positive', 'Negative', 'Neutral', 'Positive', 'Negative', 'Neutral']
}
df = pd.DataFrame(data)

print(f"Dataset Shape: {df.shape}")
print(f"Class Distribution:\n{df['sentiment'].value_counts()}")

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(clean_tokens)

df['clean_text'] = df['text'].apply(preprocess_text)
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
bow_vec = CountVectorizer()
X_train_bow = bow_vec.fit_transform(X_train)
X_test_bow = bow_vec.transform(X_test)
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

results = []

def evaluate_model(model_name, model, X_t, X_v, y_t, y_v, vectorizer_name):
    model.fit(X_t, y_t)
    y_pred = model.predict(X_v)
    
    results.append({
        "Model": model_name,
        "Vectorizer": vectorizer_name,
        "Accuracy": accuracy_score(y_v, y_pred),
        "Precision": precision_score(y_v, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_v, y_pred, average='weighted', zero_division=0),
        "F1 Score": f1_score(y_v, y_pred, average='weighted', zero_division=0)
    })

for name, model in models.items():
    evaluate_model(name, model, X_train_bow, X_test_bow, y_train, y_test, "BoW")
    evaluate_model(name, model, X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")
results_df = pd.DataFrame(results)
print("\n--- Model Performance Comparison ---")
print(results_df.sort_values(by="F1 Score", ascending=False))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


Dataset Shape: (6, 2)
Class Distribution:
sentiment
Positive    2
Negative    2
Neutral     2
Name: count, dtype: int64

--- Model Performance Comparison ---
                 Model Vectorizer  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression        BoW       0.0        0.0     0.0       0.0
1  Logistic Regression     TF-IDF       0.0        0.0     0.0       0.0
2          Naive Bayes        BoW       0.0        0.0     0.0       0.0
3          Naive Bayes     TF-IDF       0.0        0.0     0.0       0.0
4        Decision Tree        BoW       0.0        0.0     0.0       0.0
5        Decision Tree     TF-IDF       0.0        0.0     0.0       0.0
